In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

In [3]:
data_type = "drug"
name = "gdsc2"

In [4]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    # 0,  # Cell
    1  # Drug
]

load gdsc2
unique drugs: 69
unique genes: 153
DTI unique genes:  153
Top 90% variable genes:  1957
Total:  2089
Final gene exp shape: (910, 2089)
Final drug Act shape: (240, 910)


100%|██████████| 25/25 [00:01<00:00, 14.01it/s]


Done!


In [15]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, best_val_labels, best_val_prob, best_metrics, _, _, _) = No_atten_drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [16]:
method = "GCN"
params = get_params(method, f"NCI_{method}_New")
PATH = f"../{name}_data/"
params = config = {
    'dropout1': 0.5,
    'dropout2': 0.35,
    'hidden1': 512,
    'hidden2': 248,
    'hidden3': 210,
    'heads': 2,
    'activation': 'relu',
    'optimizer': 'Adam',
    'lr': 0.0002418546386305523,
    'weight_decay': 7.521978942286918e-06,
    'scheduler': 'Cosine',
    'T_max': 29,
    'gnn_layer': 'GCN'
}

params.update(
    {
        "n_drug": drug_sim.shape[0],
        "n_cell": cell_sim.shape[0],
        "n_gene": gene_sim.shape[0],
        "epochs": 1,
        "heads": 1,
        "hidden1": 128,
        "hidden2": 128,
        "hidden3": 128,
    }
)

In [17]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue
        epochs = []
        true_data_s = pd.DataFrame()
        predict_data_s = pd.DataFrame()
        for fold in range(n_kfold):
            true_data, predict_data = drGAT_new(
                res_mat=res,
                null_mask=null_mask.T.values,
                target_dim=dim,
                target_index=target_index,
                S_d=drug_sim,
                S_c=cell_sim,
                S_g=gene_sim,
                A_cg=A_cg,
                A_dg=A_dg,
                PATH=PATH,
                params=params,
                device=device,
                seed=seed,
            )

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )

0it [00:00, ?it/s]

Using device: cuda



100%|██████████| 1/1 [00:00<00:00, 13.46it/s]
4it [00:03,  1.28it/s]

Best model found at epoch 1
Using device: cuda



100%|██████████| 1/1 [00:00<00:00, 13.73it/s]
11it [00:06,  1.86it/s]

...

Best model found at epoch 1
Using device: cuda



100%|██████████| 1/1 [00:00<00:00, 13.47it/s]
235it [06:40,  1.56s/it]

Best model found at epoch 1
Using device: cuda



100%|██████████| 1/1 [00:00<00:00, 13.42it/s]
240it [06:43,  1.68s/it]

Best model found at epoch 1


In [17]:
# true_datas.to_csv(f"new_true_drug_{method}_{name}.csv")
# predict_datas.to_csv(f"new_pred_drug_{method}_{name}.csv")

In [18]:
compute_metrics_stats

In [20]:
true_datas.shape[0]

129

In [21]:
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, brier_score_loss,
                             cohen_kappa_score, f1_score, fbeta_score,
                             log_loss, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)

In [24]:
res = pd.DataFrame()
for i in range(true_datas.shape[0]):
    # データ前処理
    true_labels = true_datas.loc[i].dropna()
    pred_values = predict_datas.loc[i].dropna()
    pred_labels = np.round(pred_values).astype(int)
    
    # メトリクス計算
    metrics = {
        "ACC": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1": f1_score(true_labels, pred_labels, zero_division=0),
        "AUROC": roc_auc_score(true_labels, pred_values),
        "AUPR": average_precision_score(true_labels, pred_values),
        "MCC": matthews_corrcoef(true_labels, pred_labels),
        "Specificity": recall_score(
            true_labels, pred_labels, pos_label=0, zero_division=0
        ),
        "Balanced_ACC": balanced_accuracy_score(true_labels, pred_labels),
        "LogLoss": log_loss(true_labels, pred_values),
        "Brier": brier_score_loss(true_labels, pred_values),
    }
    res = pd.concat([res, pd.DataFrame([metrics])], ignore_index=True)

In [25]:
res

,ACC,Precision,Recall,F1,AUROC,AUPR,MCC,Specificity,Balanced_ACC,LogLoss,Brier
0,0.484375,0.488152,0.643750,0.555256,0.478340,0.508872,-0.032970,0.325000,0.484375,0.700010,0.253373
1,0.500000,0.500000,1.000000,0.666667,0.507183,0.508660,0.000000,0.000000,0.500000,0.703527,0.255110
2,0.500000,0.000000,0.000000,0.000000,0.642857,0.580294,0.000000,1.000000,0.500000,0.753349,0.278364
3,0.500000,0.000000,0.000000,0.000000,0.501730,0.524149,0.000000,1.000000,0.500000,0.701630,0.254218
4,0.522727,0.512821,0.909091,0.655738,0.442149,0.449333,0.071611,0.136364,0.522727,0.714565,0.260107
...,...,...,...,...,...,...,...,...,...,...,...
124,0.500000,0.000000,0.000000,0.000000,0.506958,0.495335,0.000000,1.000000,0.500000,0.786951,0.292423
125,0.500000,0.000000,0.000000,0.000000,0.616690,0.567900,0.000000,1.000000,0.500000,0.716329,0.261432
126,0.500000,0.000000,0.000000,0.000000,0.499408,0.506662,0.000000,1.000000,0.500000,0.709854,0.258217
127,0.500000,0.500000,0.842105,0.627451,0.434287,0.456675,0.000000,0.157895,0.500000,0.721967,0.264075
